# CodeAlpha Data Analytics Internship
## Task 3: Data Visualization — Superstore Sales Performance

- **Intern**: Om Katkar
- **Dataset**: Superstore Sales & Profit Records (`dataset/superstore_sales.csv`)
- **Libraries**: `pandas`, `matplotlib`, `seaborn`, `numpy`

---
### Executive Data Story
This notebook transforms raw transactional sales records into high-impact visual formats to reveal key business drivers: monthly revenue trajectories, category and sub-category profitability, regional sales contributions, and the margin impact of promotional discounting.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
print("Visualization libraries loaded successfully!")

### 1. Data Ingestion & Preprocessing

In [ ]:
df = pd.read_csv("dataset/superstore_sales.csv")
df["order_date"] = pd.to_datetime(df["order_date"])
df["year_month"] = df["order_date"].dt.to_period("M").dt.to_timestamp()
df["profit_margin_pct"] = (df["profit"] / df["sales"]) * 100
display(df.head())

### 2. Multi-Panel Executive Dashboard
A unified 4-panel dashboard communicating revenue trends, category breakdown, regional contribution, and discount elasticity.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Executive Sales & Profit Performance Dashboard", fontsize=18, y=0.98)

# Panel 1: Monthly Trends
monthly = df.groupby("year_month")[["sales", "profit"]].sum().reset_index()
axes[0, 0].plot(monthly["year_month"], monthly["sales"], marker="o", color="#1f77b4", linewidth=2.5, label="Monthly Sales ($)")
axes[0, 0].plot(monthly["year_month"], monthly["profit"], marker="s", color="#2ca02c", linewidth=2.5, label="Monthly Profit ($)")
axes[0, 0].fill_between(monthly["year_month"], monthly["profit"], color="#2ca02c", alpha=0.15)
axes[0, 0].set_title("A. Monthly Revenue & Profit Growth")
axes[0, 0].yaxis.set_major_formatter(ticker.StrMethodFormatter("${x:,.0f}"))
axes[0, 0].legend()

# Panel 2: Category Breakdown
cat_summary = df.groupby("category")[["sales", "profit"]].sum().reset_index()
x_pos = np.arange(len(cat_summary["category"]))
axes[0, 1].bar(x_pos - 0.17, cat_summary["sales"], width=0.35, label="Sales ($)", color="#4575b4")
axes[0, 1].bar(x_pos + 0.17, cat_summary["profit"], width=0.35, label="Profit ($)", color="#74add1")
axes[0, 1].set_title("B. Revenue & Profit by Category")
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels(cat_summary["category"])
axes[0, 1].yaxis.set_major_formatter(ticker.StrMethodFormatter("${x:,.0f}"))
axes[0, 1].legend()

# Panel 3: Regional Sales
reg_summary = df.groupby("region")[["sales", "profit"]].sum().sort_values(by="sales", ascending=True).reset_index()
axes[1, 0].barh(reg_summary["region"], reg_summary["sales"], color="#313695", alpha=0.85, label="Sales")
axes[1, 0].barh(reg_summary["region"], reg_summary["profit"], color="#fdae61", alpha=0.9, label="Profit")
axes[1, 0].set_title("C. Regional Contribution: Sales vs. Profit")
axes[1, 0].xaxis.set_major_formatter(ticker.StrMethodFormatter("${x:,.0f}"))
axes[1, 0].legend()

# Panel 4: Discount Margin Sensitivity
sns.regplot(data=df, x="discount", y="profit_margin_pct", ax=axes[1, 1],
            scatter_kws={"alpha": 0.3, "color": "#4575b4"}, line_kws={"color": "#d73027", "linewidth": 2.5})
axes[1, 1].axhline(0, color="gray", linestyle="--")
axes[1, 1].set_title("D. Discount Rate vs. Profit Margin (%)")
axes[1, 1].xaxis.set_major_formatter(ticker.PercentFormatter(1.0))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 3. Sub-Category Profitability Ranking
Diverging bar chart visualizing profitable vs loss-making product subcategories.

In [ ]:
sub_df = df.groupby(["category", "sub_category"])[["sales", "profit"]].sum().reset_index().sort_values(by="profit", ascending=True)
plt.figure(figsize=(12, 6))
colors = ["#d73027" if p < 0 else "#1a9850" for p in sub_df["profit"]]
plt.barh(sub_df["sub_category"], sub_df["profit"], color=colors)
plt.axvline(0, color="black", linewidth=1.2)
plt.title("Sub-Category Profitability Ranking")
plt.xlabel("Net Profit ($)")
plt.show()